# Run `main.py` With Tunable Parameters

Edit the `params` dict below and rerun the execute cell.

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd()
MAIN_SCRIPT = PROJECT_ROOT / "main.py"
PYTHON_BIN = Path(sys.executable)

if not MAIN_SCRIPT.exists():
    raise FileNotFoundError(f"Cannot find main.py at {MAIN_SCRIPT}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Python:       {PYTHON_BIN}")
print(f"Script:       {MAIN_SCRIPT}")

In [ ]:
# Tune these values between runs
params = {
    "log_run": False,
    "bonding_fragments": [(1, 2), (1, 3), (2, 4)],
    "target_path": "data/target.sdf",
    "box_size": "16",
    "divisions": "2",
    "num_fragments": 4,
    "compression": True,
    "q_fragment": 1.0,
    "q_clash": 1.0,
    "q_bond": 1.0,
    "clip_max": 500.0,
    "sampler": "SA",  # SA or BQM
    "rmsd_criterion": 2.0,

    "num_reads": 10,
    "num_sweeps": 5000,

    "lambda_fragment": 5,
    "lambda_clash": 1,
    "lambda_bond": 3,
    
}

params

In [ ]:
def _serialize_bonding_fragments(value):
    # main.py accepts formats like "1-2,1-3,2-4"
    if isinstance(value, str):
        return value
    return ",".join(f"{a}-{b}" for a, b in value)


def build_main_command(params_dict):
    cmd = [str(PYTHON_BIN), str(MAIN_SCRIPT)]

    bool_flags = {
        "log_run": ("--log-run", "--no-log-run"),
        "compression": ("--compression", "--no-compression"),
    }

    for key, value in params_dict.items():
        if key in bool_flags:
            true_flag, false_flag = bool_flags[key]
            cmd.append(true_flag if bool(value) else false_flag)
            continue

        flag = f"--{key.replace('_', '-')}"
        if key == "bonding_fragments":
            value = _serialize_bonding_fragments(value)
        cmd.extend([flag, str(value)])

    return cmd


cmd = build_main_command(params)
print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

In [ ]:
# Execute main.py once with current params
cmd = build_main_command(params)

start = time.time()
result = subprocess.run(
    cmd,
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
elapsed = time.time() - start

print(f"Exit code: {result.returncode}")
print(f"Elapsed:   {elapsed:.2f}s")

if result.stdout:
    print("\n=== STDOUT ===")
    print(result.stdout)

if result.stderr:
    print("\n=== STDERR ===")
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("main.py failed. See STDERR above.")

In [ ]:
# Optional: simple sweep template
sweep = [
    {"lambda_fragment": 1.0, "lambda_clash": 1.0, "lambda_bond": 10.0},
    {"lambda_fragment": 1.0, "lambda_clash": 5.0, "lambda_bond": 5.0},
]

for idx, overrides in enumerate(sweep, start=1):
    run_params = {**params, **overrides}
    cmd = build_main_command(run_params)
    print(f"\n--- Sweep run {idx} ---")
    print(" ".join(shlex.quote(x) for x in cmd))

    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(f"Exit code: {result.returncode}")
    if result.returncode != 0:
        print(result.stderr)
        break

In [ ]:
# Run main.py N times and store metrics in NumPy arrays (parallel)
import concurrent.futures as cf
from pathlib import Path
import re
import shutil
import tempfile
import numpy as np

N_RUNS = 10
P_PARALLEL = 4  # number of concurrent subprocesses
run_params = dict(params)

qubo_pattern = re.compile(r"qubo_score=([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")
rmsd_pattern = re.compile(r"Heavy atom RMSD \(symmetry-aware best\):\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")
centroid_pattern = re.compile(r"Heavy atom centroid distance:\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")
shape_sim_pattern = re.compile(r"Heavy atom shape Tanimoto similarity:\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")
shape_dist_pattern = re.compile(r"Heavy atom shape Tanimoto distance:\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)")


def extract_metric(stdout, pattern, label):
    match = pattern.search(stdout)
    if not match:
        raise ValueError(f"Could not parse '{label}' from main.py output.")
    return float(match.group(1))


def run_main_once(run_idx):
    cmd = build_main_command(run_params)

    # main.py writes output files relative to CWD; isolate each run to avoid collisions.
    with tempfile.TemporaryDirectory(prefix=f"main_run_{run_idx:03d}_") as tmpdir:
        run_cwd = Path(tmpdir)
        data_link = run_cwd / "data"
        try:
            data_link.symlink_to(PROJECT_ROOT / "data", target_is_directory=True)
        except OSError:
            shutil.copytree(PROJECT_ROOT / "data", data_link)

        result = subprocess.run(cmd, cwd=run_cwd, text=True, capture_output=True)

    if result.returncode != 0:
        return {
            "run_idx": run_idx,
            "returncode": result.returncode,
            "stdout": result.stdout,
            "stderr": result.stderr,
        }

    stdout = result.stdout
    return {
        "run_idx": run_idx,
        "returncode": 0,
        "qubo": extract_metric(stdout, qubo_pattern, "QUBO score"),
        "rmsd": extract_metric(stdout, rmsd_pattern, "Heavy atom RMSD"),
        "centroid": extract_metric(stdout, centroid_pattern, "Heavy atom centroid distance"),
        "shape_sim": extract_metric(stdout, shape_sim_pattern, "Heavy atom shape Tanimoto similarity"),
        "shape_dist": extract_metric(stdout, shape_dist_pattern, "Heavy atom shape Tanimoto distance"),
    }


max_workers = max(1, min(int(P_PARALLEL), N_RUNS))
results = [None] * N_RUNS

with cf.ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_run = {
        executor.submit(run_main_once, run_idx): run_idx
        for run_idx in range(1, N_RUNS + 1)
    }

    for future in cf.as_completed(future_to_run):
        run_idx = future_to_run[future]
        payload = future.result()

        if payload["returncode"] != 0:
            print(payload["stdout"])
            print(payload["stderr"])
            raise RuntimeError(
                f"main.py failed on run {run_idx} with exit code {payload['returncode']}"
            )

        results[run_idx - 1] = payload
        print(
            f"Run {run_idx:03d}/{N_RUNS}: "
            f"qubo={payload['qubo']:.4f}, rmsd={payload['rmsd']:.4f}, "
            f"centroid={payload['centroid']:.4f}, sim={payload['shape_sim']:.4f}, dist={payload['shape_dist']:.4f}"
        )


qubo_scores = np.asarray([item["qubo"] for item in results], dtype=float)
heavy_atom_rmsd = np.asarray([item["rmsd"] for item in results], dtype=float)
heavy_atom_centroid_distance = np.asarray([item["centroid"] for item in results], dtype=float)
heavy_atom_shape_tanimoto_similarity = np.asarray([item["shape_sim"] for item in results], dtype=float)
heavy_atom_shape_tanimoto_distance = np.asarray([item["shape_dist"] for item in results], dtype=float)

print("\nArray shapes:")
print("qubo_scores:", qubo_scores.shape)
print("heavy_atom_rmsd:", heavy_atom_rmsd.shape)
print("heavy_atom_centroid_distance:", heavy_atom_centroid_distance.shape)
print("heavy_atom_shape_tanimoto_similarity:", heavy_atom_shape_tanimoto_similarity.shape)
print("heavy_atom_shape_tanimoto_distance:", heavy_atom_shape_tanimoto_distance.shape)
